In [1]:
import folium
from openlocationcode import openlocationcode as olc

# ---------------------------------------------------------
# REFERÊNCIA DA REGIÃO
# Necessária para expandir Plus Codes curtos
# ---------------------------------------------------------
ref_lat = -23.5560
ref_lon = -46.6450

# ---------------------------------------------------------
# FUNÇÃO: PLUS CODE -> COORDENADAS
# ---------------------------------------------------------
def plus_code_para_coords(codigo_curto, ref_lat, ref_lon):
    codigo_completo = olc.recoverNearest(codigo_curto, ref_lat, ref_lon)
    area = olc.decode(codigo_completo)
    lat = area.latitudeCenter
    lon = area.longitudeCenter
    return (lat, lon), codigo_completo

# ---------------------------------------------------------
# UNIDADES
# ---------------------------------------------------------
unidades = [
    {
        "nome": "Hospital Leforte Liberdade",
        "tipo": "Hospital Geral",
        "plus_code": "C9R8+R3"
    },
    {
        "nome": "Hospital Alemão Oswaldo Cruz",
        "tipo": "Hospital Privado",
        "plus_code": "C9J4+7Q"
    },
    {
        "nome": "Hospital Pró Matre Paulista",
        "tipo": "Maternidade",
        "plus_code": "C9M2+R2"
    },
    {
        "nome": "Hospital TotalCor",
        "tipo": "Hospital Cardiológico",
        "plus_code": "C8MX+2M"
    },
    {
        "nome": "Hospital Samaritano Paulista",
        "tipo": "Hospital Geral",
        "plus_code": "C9M3+X7"
    },
    {
        "nome": "Hospital IGESP Paulista",
        "tipo": "Hospital Geral",
        "plus_code": "C8QX+PP"
    },
    {
        "nome": "HCor - Hospital do Coração",
        "tipo": "Hospital Cardiológico",
        "plus_code": "C9G4+RG"
    },
    {
        "nome": "Hospital Luz Vila Mariana",
        "tipo": "Hospital Geral",
        "plus_code": "C995+CV"
    },
    {
        "nome": "UBS Humaitá",
        "tipo": "UBS / Posto de Saúde",
        "plus_code": "C9V5+66"
    },
    {
        "nome": "Hospital Municipal Infantil Menino Jesus",
        "tipo": "Hospital Infantil",
        "plus_code": "C9Q3+W2"
    },
    {
        "nome": "UBS Nossa Senhora do Brasil",
        "tipo": "UBS / Posto de Saúde",
        "plus_code": "C9P2+W7"
    },
    {
        "nome": "Hospital Santa Catarina - Paulista",
        "tipo": "Hospital",
        "plus_code": "C9J3+2Q"
    }
]

# ---------------------------------------------------------
# CONVERTE PLUS CODE PARA COORDENADAS
# ---------------------------------------------------------
for unidade in unidades:
    coords, plus_code_completo = plus_code_para_coords(
        unidade["plus_code"],
        ref_lat,
        ref_lon
    )
    unidade["coords"] = coords
    unidade["plus_code_completo"] = plus_code_completo

# ---------------------------------------------------------
# SEDE PROVISÓRIA
# Se quiser depois, trocamos por um Plus Code seu também
# ---------------------------------------------------------
sede = {
    "nome": "Hub Logístico Saúde Centro",
    "tipo": "Sede de início das entregas",
    "coords": (-23.55052, -46.63331)
}

# ---------------------------------------------------------
# CRIA O MAPA
# ---------------------------------------------------------
mapa = folium.Map(
    location=ref_lat and [ref_lat, ref_lon],
    zoom_start=14
)

# ---------------------------------------------------------
# MARCADOR DA SEDE
# ---------------------------------------------------------
popup_sede = f"""
<b>{sede['nome']}</b><br>
Tipo: {sede['tipo']}
"""

folium.Marker(
    location=sede["coords"],
    tooltip=sede["nome"],
    popup=folium.Popup(popup_sede, max_width=280),
    icon=folium.Icon(color="green", icon="home")
).add_to(mapa)

# ---------------------------------------------------------
# MARCADORES DAS UNIDADES
# ---------------------------------------------------------
for unidade in unidades:
    cor = "blue" if "UBS" in unidade["tipo"] else "red"

    popup_html = f"""
    <b>{unidade['nome']}</b><br>
    Tipo: {unidade['tipo']}<br>
    Plus Code: {unidade['plus_code']}<br>
    Plus Code completo: {unidade['plus_code_completo']}<br>
    Coordenadas: {unidade['coords'][0]:.6f}, {unidade['coords'][1]:.6f}
    """

    folium.Marker(
        location=unidade["coords"],
        tooltip=unidade["nome"],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(color=cor)
    ).add_to(mapa)

mapa


## ⚠️ Não executar a célula abaixo

Ela calcula todas as combinações de rotas para **12 unidades**, totalizando **479.001.600 rotas**.


In [2]:
from itertools import permutations

import networkx as nx
import osmnx as ox

# ---------------------------------------------------------
# TESTE DE MELHOR TRAJETO PELAS RUAS
# Sempre inicia na sede e visita todas as unidades pela menor rota
# usando a malha viaria do OpenStreetMap.
# ---------------------------------------------------------

ox.settings.use_cache = True
ox.settings.log_console = False

pontos_entrega = [sede] + unidades
centro_mapa = (
    sum(ponto["coords"][0] for ponto in pontos_entrega) / len(pontos_entrega),
    sum(ponto["coords"][1] for ponto in pontos_entrega) / len(pontos_entrega)
)

maior_distancia_graus = max(
    ((ponto["coords"][0] - centro_mapa[0]) ** 2 + (ponto["coords"][1] - centro_mapa[1]) ** 2) ** 0.5
    for ponto in pontos_entrega
)
raio_busca_m = int(maior_distancia_graus * 111_000) + 2_000

G = ox.graph_from_point(
    centro_mapa,
    dist=raio_busca_m,
    network_type="drive",
    simplify=True
)

for ponto in pontos_entrega:
    lat, lon = ponto["coords"]
    ponto["no_rua"] = ox.distance.nearest_nodes(G, X=lon, Y=lat)

pares_distancias = {}

for origem in pontos_entrega:
    for destino in pontos_entrega:
        if origem == destino:
            continue

        pares_distancias[(origem["nome"], destino["nome"])] = nx.shortest_path_length(
            G,
            origem["no_rua"],
            destino["no_rua"],
            weight="length"
        )


def distancia_rota_por_ruas(rota):
    pontos = [sede] + list(rota)
    return sum(
        pares_distancias[(pontos[i]["nome"], pontos[i + 1]["nome"])]
        for i in range(len(pontos) - 1)
    )


menor_rota = min(
    permutations(unidades),
    key=distancia_rota_por_ruas
)

menor_distancia_m = distancia_rota_por_ruas(menor_rota)
pontos_rota = [sede] + list(menor_rota)

coordenadas_rota = []

for origem, destino in zip(pontos_rota, pontos_rota[1:]):
    trecho = nx.shortest_path(
        G,
        origem["no_rua"],
        destino["no_rua"],
        weight="length"
    )

    if coordenadas_rota:
        trecho = trecho[1:]

    coordenadas_rota.extend(
        (G.nodes[no]["y"], G.nodes[no]["x"])
        for no in trecho
    )

mapa_rota = folium.Map(
    location=sede["coords"],
    zoom_start=13
)

folium.Marker(
    location=sede["coords"],
    tooltip="0 - Sede",
    popup=folium.Popup(f"<b>0 - {sede['nome']}</b><br>Ponto inicial", max_width=280),
    icon=folium.Icon(color="green", icon="home")
).add_to(mapa_rota)

for ordem, unidade in enumerate(menor_rota, start=1):
    folium.Marker(
        location=unidade["coords"],
        tooltip=f"{ordem} - {unidade['nome']}",
        popup=folium.Popup(
            f"<b>{ordem} - {unidade['nome']}</b><br>Tipo: {unidade['tipo']}",
            max_width=350
        ),
        icon=folium.Icon(color="orange", icon="flag")
    ).add_to(mapa_rota)

folium.PolyLine(
    locations=coordenadas_rota,
    color="purple",
    weight=5,
    opacity=0.85,
    tooltip=f"Menor rota pelas ruas: {menor_distancia_m / 1000:.2f} km"
).add_to(mapa_rota)

print("Melhor ordem de visita pelas ruas:")
print(f"0. {sede['nome']} (sede)")

for ordem, unidade in enumerate(menor_rota, start=1):
    print(f"{ordem}. {unidade['nome']} - {unidade['tipo']}")

print(f"\nDistancia total estimada pelas ruas: {menor_distancia_m / 1000:.2f} km")

mapa_rota


KeyboardInterrupt: 

In [ ]:
from functools import lru_cache
import unicodedata

import folium
from folium.plugins import AntPath
import networkx as nx
import osmnx as ox

# ---------------------------------------------------------
# ENTREGA AVULSA: ESCOLHA OS LOCAIS DA ENTREGA
# Edite apenas esta lista com os locais que quer visitar.
# Pode usar o nome completo ou um trecho unico do nome.
# A rota sempre sai da sede e segue pelas ruas.
# ---------------------------------------------------------

locais_entrega = [
    "Hospital IGESP Paulista",
    "HCor - Hospital do Coração",
    "Hospital Luz Vila Mariana",
]


def normalizar_texto(texto):
    texto_sem_acento = unicodedata.normalize("NFKD", texto)
    texto_sem_acento = "".join(
        caractere for caractere in texto_sem_acento
        if not unicodedata.combining(caractere)
    )
    return texto_sem_acento.casefold().strip()


def buscar_unidade(nome_local):
    nome_normalizado = normalizar_texto(nome_local)
    unidades_por_nome = {
        normalizar_texto(unidade["nome"]): unidade
        for unidade in unidades
    }

    if nome_normalizado in unidades_por_nome:
        return unidades_por_nome[nome_normalizado]

    candidatas = [
        unidade for unidade in unidades
        if nome_normalizado in normalizar_texto(unidade["nome"])
    ]

    if len(candidatas) == 1:
        return candidatas[0]

    if not candidatas:
        nomes_disponiveis = "\n".join(f"- {unidade['nome']}" for unidade in unidades)
        raise ValueError(
            f"Local nao encontrado: {nome_local}\n\n"
            f"Locais disponiveis:\n{nomes_disponiveis}"
        )

    nomes_candidatos = "\n".join(f"- {unidade['nome']}" for unidade in candidatas)
    raise ValueError(
        f"O texto '{nome_local}' encontrou mais de uma unidade. "
        f"Use um nome mais especifico:\n{nomes_candidatos}"
    )


def montar_grafo_por_ruas(pontos):
    centro_mapa = (
        sum(ponto["coords"][0] for ponto in pontos) / len(pontos),
        sum(ponto["coords"][1] for ponto in pontos) / len(pontos)
    )

    maior_distancia_graus = max(
        ((ponto["coords"][0] - centro_mapa[0]) ** 2 + (ponto["coords"][1] - centro_mapa[1]) ** 2) ** 0.5
        for ponto in pontos
    )
    raio_busca_m = int(maior_distancia_graus * 111_000) + 2_000

    return ox.graph_from_point(
        centro_mapa,
        dist=raio_busca_m,
        network_type="drive",
        simplify=True
    )


def comprimento_caminho_m(grafo_ruas, caminho):
    distancia = 0

    for origem, destino in zip(caminho, caminho[1:]):
        arestas = grafo_ruas.get_edge_data(origem, destino).values()
        distancia += min(aresta.get("length", 0) for aresta in arestas)

    return distancia


def calcular_melhor_rota_entrega(nomes_locais):
    if not nomes_locais:
        raise ValueError("Informe pelo menos um local em locais_entrega.")

    unidades_entrega = [buscar_unidade(nome) for nome in nomes_locais]
    pontos = [sede] + unidades_entrega

    ox.settings.use_cache = True
    ox.settings.log_console = False

    grafo_ruas = montar_grafo_por_ruas(pontos)

    for ponto in pontos:
        lat, lon = ponto["coords"]
        ponto["no_rua_entrega"] = ox.distance.nearest_nodes(grafo_ruas, X=lon, Y=lat)

    distancias = {}
    caminhos = {}

    for origem in pontos:
        for destino in pontos:
            if origem == destino:
                continue

            chave = (origem["nome"], destino["nome"])
            caminho = nx.shortest_path(
                grafo_ruas,
                origem["no_rua_entrega"],
                destino["no_rua_entrega"],
                weight="length"
            )
            caminhos[chave] = caminho
            distancias[chave] = comprimento_caminho_m(grafo_ruas, caminho)

    nomes_unidades = tuple(unidade["nome"] for unidade in unidades_entrega)

    @lru_cache(None)
    def melhor_caminho(origem_nome, restantes):
        if not restantes:
            return 0, []

        melhor_distancia = float("inf")
        melhor_ordem = []

        for proximo_nome in restantes:
            proximos_restantes = tuple(
                nome for nome in restantes
                if nome != proximo_nome
            )
            distancia_restante, ordem_restante = melhor_caminho(proximo_nome, proximos_restantes)
            distancia_total = distancias[(origem_nome, proximo_nome)] + distancia_restante

            if distancia_total < melhor_distancia:
                melhor_distancia = distancia_total
                melhor_ordem = [proximo_nome] + ordem_restante

        return melhor_distancia, melhor_ordem

    menor_distancia_m, melhor_ordem_nomes = melhor_caminho(sede["nome"], nomes_unidades)
    unidades_por_nome = {unidade["nome"]: unidade for unidade in unidades_entrega}
    melhor_rota = [unidades_por_nome[nome] for nome in melhor_ordem_nomes]
    pontos_rota = [sede] + melhor_rota

    coordenadas_rota = []

    for origem, destino in zip(pontos_rota, pontos_rota[1:]):
        trecho = caminhos[(origem["nome"], destino["nome"])]

        if coordenadas_rota:
            trecho = trecho[1:]

        coordenadas_rota.extend(
            (grafo_ruas.nodes[no]["y"], grafo_ruas.nodes[no]["x"])
            for no in trecho
        )

    mapa_entrega = folium.Map(
        location=sede["coords"],
        zoom_start=13
    )

    folium.Marker(
        location=sede["coords"],
        tooltip="0 - Sede",
        popup=folium.Popup(f"<b>0 - {sede['nome']}</b><br>Ponto inicial", max_width=280),
        icon=folium.Icon(color="green", icon="home")
    ).add_to(mapa_entrega)

    for ordem, unidade in enumerate(melhor_rota, start=1):
        folium.Marker(
            location=unidade["coords"],
            tooltip=f"{ordem} - {unidade['nome']}",
            popup=folium.Popup(
                f"<b>{ordem} - {unidade['nome']}</b><br>Tipo: {unidade['tipo']}",
                max_width=350
            ),
            icon=folium.Icon(color="orange", icon="flag")
        ).add_to(mapa_entrega)

    folium.PolyLine(
        locations=coordenadas_rota,
        color="purple",
        weight=3,
        opacity=0.35,
        tooltip=f"Menor rota de entrega: {menor_distancia_m / 1000:.2f} km"
    ).add_to(mapa_entrega)

    AntPath(
        locations=coordenadas_rota,
        color="purple",
        pulse_color="white",
        weight=6,
        opacity=0.9,
        delay=700,
        tooltip=f"Rota animada: {menor_distancia_m / 1000:.2f} km"
    ).add_to(mapa_entrega)

    print("Melhor ordem para esta entrega:")
    print(f"0. {sede['nome']} (sede)")

    for ordem, unidade in enumerate(melhor_rota, start=1):
        print(f"{ordem}. {unidade['nome']} - {unidade['tipo']}")

    print(f"\nDistancia total estimada pelas ruas: {menor_distancia_m / 1000:.2f} km")

    return mapa_entrega


calcular_melhor_rota_entrega(locais_entrega)


In [ ]:
from route_engine import exibir_mapa_unidades

exibir_mapa_unidades()


In [ ]:
from route_engine import calcular_melhor_rota_entrega, criar_mapa_rota

locais_entrega = [
    "Hospital IGESP Paulista",
    "HCor - Hospital do Coração",
    "Hospital Luz Vila Mariana",
]

resultado = calcular_melhor_rota_entrega(locais_entrega)
mapa = criar_mapa_rota(resultado)

mapa

## Exemplo: algoritmo genético com restrições operacionais

Este exemplo calcula uma rota considerando **prioridade**, **capacidade de carga** e **autonomia do veículo**. Quando uma restrição impede que todas as entregas continuem na mesma saída, o engine cria uma nova viagem a partir da sede.

Os parâmetros reduzidos de população e gerações deixam a demonstração mais rápida no notebook.

In [3]:
from route_engine import calcular_melhor_rota_entrega

pontos_teste = [
    "Hospital Alemão Oswaldo Cruz",
    "Hospital Pró Matre Paulista",
    "HCor - Hospital do Coração",
    "Hospital Luz Vila Mariana",
    "UBS Humaitá",
    "Hospital Santa Catarina - Paulista",
]

resultado_genetico = calcular_melhor_rota_entrega(
    pontos_teste,
    capacidade_kg=80,
    autonomia_km=25,
    prioridades={
        "HCor - Hospital do Coração": 1,
        "Hospital Alemão Oswaldo Cruz": 1,
        "Hospital Pró Matre Paulista": 1,
        "Hospital Santa Catarina - Paulista": 2,
        "Hospital Luz Vila Mariana": 2,
        "UBS Humaitá": 3,
    },
    demandas_kg={
        "Hospital Alemão Oswaldo Cruz": 30,
        "Hospital Pró Matre Paulista": 25,
        "HCor - Hospital do Coração": 35,
        "Hospital Luz Vila Mariana": 25,
        "UBS Humaitá": 15,
        "Hospital Santa Catarina - Paulista": 35,
    },
    configuracao_ga={
        "tamanho_populacao": 80,
        "geracoes": 150,
        "semente": 42,
    },
)

In [4]:
print("Algoritmo:", resultado_genetico["algoritmo"]["nome"])
print("Distância total: %.2f km" % resultado_genetico["distancia_total_km"])
print("Quantidade de viagens:", resultado_genetico["total_viagens"])
print("Demanda total: %.1f kg" % resultado_genetico["restricoes"]["demanda_total_kg"])
print()

for viagem in resultado_genetico["viagens"]:
    print(
        f"Viagem {viagem['numero']} - "
        f"{viagem['carga_kg']:.1f} kg - "
        f"{viagem['distancia_km']:.2f} km"
    )

    for parada in viagem["paradas"]:
        print(
            f"  {parada['ordem_na_viagem']}. {parada['nome']} "
            f"(prioridade {parada['prioridade_rotulo']}, "
            f"{parada['demanda_kg']:.1f} kg)"
        )

    print("  Retorno à sede")
    print()

Algoritmo: algoritmo_genetico
Distância total: 25.50 km
Quantidade de viagens: 3
Demanda total: 165.0 kg

Viagem 1 - 65.0 kg - 7.52 km
  1. HCor - Hospital do Coração (prioridade alta, 35.0 kg)
  2. Hospital Alemão Oswaldo Cruz (prioridade alta, 30.0 kg)
  Retorno à sede

Viagem 2 - 60.0 kg - 8.11 km
  1. Hospital Pró Matre Paulista (prioridade alta, 25.0 kg)
  2. Hospital Santa Catarina - Paulista (prioridade media, 35.0 kg)
  Retorno à sede

Viagem 3 - 40.0 kg - 9.87 km
  1. Hospital Luz Vila Mariana (prioridade media, 25.0 kg)
  2. UBS Humaitá (prioridade normal, 15.0 kg)
  Retorno à sede



In [5]:
from route_engine import criar_mapa_rota

criar_mapa_rota(resultado_genetico)